In [14]:
import pandas as pd
import json
import random
import os

# 1. DOSSIERS
os.makedirs("data/processed", exist_ok=True)

# 2. CHARGEMENT DE TON LEXIQUE (522 MOTS)
chatbot_data = []
try:
    local_df = pd.read_csv('mon_lexique.csv')
    c1, c2 = local_df.columns[0], local_df.columns[1]
    for _, row in local_df.iterrows():
        chatbot_data.append({
            "instruction": f"Comment dit-on en Dioula : {row[c1]} ?",
            "response": f"En Dioula, on dit : {row[c2]}."
        })
    print(f"✅ Lexique local : {len(chatbot_data)} mots ajoutés.")
except Exception as e:
    print(f"❌ Erreur locale : {e}")

# 3. CHARGEMENT MASAKHANE (Format Parquet - Le plus fiable)
print("--- Tentative Masakhane via Parquet ---")
try:
    # URL vers le stockage Parquet de Hugging Face pour fra-bam
    url_parquet = "https://huggingface.co/datasets/masakhane/mafand/resolve/main/data/fra-bam/train.parquet"
    df_m = pd.read_parquet(url_parquet)

    count_m = 0
    for _, row in df_m.iterrows():
        # On extrait selon la structure Parquet (souvent une colonne 'translation' contenant un dict)
        if 'translation' in row:
            f = row['translation']['fra']
            b = row['translation']['bam']
        else:
            f, b = row['source_sentence'], row['target_sentence']

        chatbot_data.append({
            "instruction": f"Comment dit-on en Dioula : {f} ?",
            "response": f"En Dioula, on dit : {b}."
        })
        count_m += 1
    print(f"✅ Masakhane : {count_m} phrases ajoutées.")
except Exception as e:
    print(f"⚠️ Échec Parquet : {e}. On tente le CSV brut...")
    try:
        url_csv = "https://huggingface.co/datasets/masakhane/mafand/resolve/main/data/fra-bam/train.csv"
        df_c = pd.read_csv(url_csv)
        for _, row in df_c.iterrows():
            chatbot_data.append({"instruction": f"Comment dit-on en Dioula : {row[0]} ?", "response": f"En Dioula, on dit : {row[1]}."})
        print(f"✅ Masakhane (via CSV) ajouté !")
    except:
        print("❌ Masakhane indisponible. On reste sur tes 522 mots.")

# 4. SAUVEGARDE
if chatbot_data:
    random.shuffle(chatbot_data)
    with open("data/processed/final_chatbot_data.jsonl", "w", encoding="utf-8") as f:
        for entry in chatbot_data:
            json.dump(entry, f, ensure_ascii=False)
            f.write("\n")
    print(f"\n🚀 TOTAL FINAL : {len(chatbot_data)} lignes prêtes !")

✅ Lexique local : 522 mots ajoutés.
--- Tentative Masakhane via Parquet ---
⚠️ Échec Parquet : HTTP Error 404: Not Found. On tente le CSV brut...
❌ Masakhane indisponible. On reste sur tes 522 mots.

🚀 TOTAL FINAL : 522 lignes prêtes !


In [ ]:
from datasets import load_dataset
from transformers import T5Tokenizer

# 1. Chargement de TON dataset
dataset = load_dataset('json', data_files='data/processed/final_chatbot_data.jsonl', split='train')

# 2. Chargement du Tokenizer de Flan-T5
model_id = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_id)

# 3. Fonction de préparation des données
def preprocess_function(examples):
    inputs = examples["instruction"]
    targets = examples["response"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Application au dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)
print("Dataset prêt pour l'entraînement !")